# Phase 4B: Full Dataset Inference
## Generate Results for ALL Videos (Phase 5-7 Pipeline)

This notebook runs inference on **ALL videos** in the dataset to generate comprehensive results for the downstream phases.

---

### Pipeline Context
```
Phase 4 Strategy:
├── Run 1: Test Set Only → "test_results.json" (Phase4_Inference_Deployment.ipynb)
│   └── Use for: Accuracy, AUC, Precision, Recall metrics
│
└── Run 2: ALL Videos → "detailed_results.json" (THIS NOTEBOOK)
    └── Use for: Phase 5-7 captioning demonstration
```

---

### Why Process All Videos?
- **Test set only**: ~190 videos (insufficient for demonstration)
- **All videos**: ~1,900 videos across 13 anomaly classes + Normal
- Phase 5 needs diverse examples: **10 videos × 13 classes = 130 demonstrations**

---

### Output Format: DICT (keyed by video_name)
```python
# detailed_results.json structure
{
    "Fighting001_x264": {
        "video_name": "Fighting001_x264",
        "video_score": 0.94,
        "is_anomaly": true,
        "segment_scores": [...],  # 16 values
        "smoothed_scores": [...], # 16 values
        "anomaly_segments": [10, 11, 12, 13],
        "max_segment_score": 0.96,
        "ground_truth": 1,
        "class_name": "Fighting",
        "original_clips": 45
    },
    "Explosion003_x264": {...},
    ...
}
```

**Why DICT format?**
- ✅ Easier to access: `results["Fighting001_x264"]`
- ✅ No duplicates (single entry per video)
- ✅ Phase 5 iterates with `for video_name, data in results.items()`

---
## Section 1: Environment Setup
---

In [ ]:
""" Phase 4B: Full Dataset Inference --> Environment Setup and Imports """

import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from typing import List, Dict, Optional, Tuple, Union
import json
import glob
from tqdm.notebook import tqdm
from scipy.ndimage import gaussian_filter1d
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# ==================== GPU Setup ====================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("="*70)
print("PHASE 4B: FULL DATASET INFERENCE")
print("="*70)
print(f"\n🎯 Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print("="*70)

PHASE 4B: FULL DATASET INFERENCE

🎯 Using device: cuda
   GPU: Tesla T4
   Memory: 14.56 GB


---
## Section 2: Model Architecture & Loading
---

In [ ]:
"""
Configuration for Phase 4B: Full Dataset Inference
"""

# ==================== Paths ====================
MODELS_PATH = "/kaggle/input/mil-model/pytorch/default/1/MIL_Models"
FEATURES_PATH = "/kaggle/input/timesformer-features/TimeSformer_Features"  # ALL features
INFERENCE_OUTPUT_PATH = r"/kaggle/working/Inference_Results_All"

# ==================== Model Parameters (Must match Phase 3) ====================
FEATURE_DIM = 768
HIDDEN_DIM_1 = 512
HIDDEN_DIM_2 = 128
DROPOUT_RATE = 0.6
TOP_K = 3
NUM_SEGMENTS = 16

# ==================== Inference Parameters ====================
ANOMALY_THRESHOLD = 0.5
SMOOTHING_SIGMA = 2

# Create output directory
os.makedirs(INFERENCE_OUTPUT_PATH, exist_ok=True)

print("\n" + "="*70)
print("CONFIGURATION")
print("="*70)
print(f"\n📁 Paths:")
print(f"   Models: {MODELS_PATH}")
print(f"   ALL Features: {FEATURES_PATH}")
print(f"   Output: {INFERENCE_OUTPUT_PATH}")
print(f"\n🔢 Model Parameters:")
print(f"   Feature Dim: {FEATURE_DIM}")
print(f"   Hidden Dims: {HIDDEN_DIM_1} → {HIDDEN_DIM_2}")
print(f"   Top-k: {TOP_K}")
print(f"   Segments: {NUM_SEGMENTS}")
print(f"\n⚙️  Inference Settings:")
print(f"   Anomaly Threshold: {ANOMALY_THRESHOLD}")
print(f"   Smoothing Sigma: {SMOOTHING_SIGMA}")
print("="*70)


CONFIGURATION

📁 Paths:
   Models: /kaggle/input/mil-model/pytorch/default/1/MIL_Models
   ALL Features: /kaggle/input/timesformer-features/TimeSformer_Features
   Output: /kaggle/working/Inference_Results_All

🔢 Model Parameters:
   Feature Dim: 768
   Hidden Dims: 512 → 128
   Top-k: 3
   Segments: 16

⚙️  Inference Settings:
   Anomaly Threshold: 0.5
   Smoothing Sigma: 2


In [ ]:
"""
MIL Network Architecture (Same as Phase 3 & 4)
"""

class MILNetwork(nn.Module):
    """
    Multiple Instance Learning Network for Video Anomaly Detection.
    Architecture: 768 → 512 → 128 → 1
    """

    def __init__(
        self,
        feature_dim: int = FEATURE_DIM,
        hidden_dim_1: int = HIDDEN_DIM_1,
        hidden_dim_2: int = HIDDEN_DIM_2,
        dropout_rate: float = DROPOUT_RATE,
        top_k: int = TOP_K
    ):
        super(MILNetwork, self).__init__()
        self.top_k = top_k

        self.fc1 = nn.Linear(feature_dim, hidden_dim_1)
        self.fc2 = nn.Linear(hidden_dim_1, hidden_dim_2)
        self.fc3 = nn.Linear(hidden_dim_2, 1)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        batch_size, num_segments, _ = x.shape
        x = x.view(-1, x.shape[-1])

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = torch.sigmoid(self.fc3(x))

        segment_scores = x.view(batch_size, num_segments)
        topk_scores, _ = torch.topk(segment_scores, k=self.top_k, dim=1)
        video_scores = topk_scores.mean(dim=1)

        return video_scores, segment_scores


print("✅ MIL Network architecture defined!")

✅ MIL Network architecture defined!


In [ ]:
"""
Load Trained Model from Phase 3
"""

def load_trained_model(model_path: str = None) -> nn.Module:
    """Load the trained MIL model."""
    print("\n" + "="*70)
    print("LOADING TRAINED MODEL")
    print("="*70)

    if model_path is None:
        best_model_path = os.path.join(MODELS_PATH, 'best_model.pth')
        final_model_path = os.path.join(MODELS_PATH, 'final_model.pth')

        if os.path.exists(best_model_path):
            model_path = best_model_path
        elif os.path.exists(final_model_path):
            model_path = final_model_path
        else:
            raise FileNotFoundError(f"No trained model found in {MODELS_PATH}")

    print(f"\n📂 Loading model from: {model_path}")

    checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)

    model = MILNetwork(
        feature_dim=FEATURE_DIM,
        hidden_dim_1=HIDDEN_DIM_1,
        hidden_dim_2=HIDDEN_DIM_2,
        dropout_rate=DROPOUT_RATE,
        top_k=TOP_K
    ).to(DEVICE)

    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)

    model.eval()

    if 'epoch' in checkpoint:
        print(f"\n📊 Training Info:")
        print(f"   Best Epoch: {checkpoint.get('epoch', 'N/A')}")
        if 'val_auc' in checkpoint:
            print(f"   Val AUC: {checkpoint['val_auc']:.4f}")

    print(f"\n✅ Model loaded successfully!")
    print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print("="*70)

    return model


# Load the model
model = load_trained_model()


LOADING TRAINED MODEL

📂 Loading model from: /kaggle/input/mil-model/pytorch/default/1/MIL_Models/best_model.pth

📊 Training Info:
   Best Epoch: 62
   Val AUC: 0.9527

✅ Model loaded successfully!
   Parameters: 459,521


---
## Section 3: Anomaly Detector Class
---

In [ ]:
"""
Anomaly Detector - Inference Wrapper
"""

class AnomalyDetector:
    """Anomaly Detection inference wrapper for the MIL model."""

    def __init__(
        self,
        model: nn.Module,
        device: torch.device = DEVICE,
        threshold: float = ANOMALY_THRESHOLD,
        smoothing_sigma: float = SMOOTHING_SIGMA,
        num_segments: int = NUM_SEGMENTS
    ):
        self.model = model
        self.device = device
        self.threshold = threshold
        self.smoothing_sigma = smoothing_sigma
        self.num_segments = num_segments
        self.model.eval()

    def preprocess_features(self, features: np.ndarray) -> torch.Tensor:
        """Preprocess features using LINEAR INTERPOLATION (matches Phase 3)."""
        if features.ndim == 1:
            segment_features = np.tile(features[np.newaxis, :], (self.num_segments, 1))
            tensor = torch.FloatTensor(segment_features).unsqueeze(0)
            return tensor.to(self.device)

        if features.ndim != 2:
            raise ValueError(f"Expected 1D or 2D features, got shape {features.shape}")

        if features.shape[0] == self.num_segments:
            tensor = torch.FloatTensor(features).unsqueeze(0)
            return tensor.to(self.device)

        # Linear interpolation
        segment_features = self._interpolate_to_segments(features)
        tensor = torch.FloatTensor(segment_features).unsqueeze(0)
        return tensor.to(self.device)

    def _interpolate_to_segments(self, features: np.ndarray) -> np.ndarray:
        """Interpolate variable-length features to fixed segments."""
        feat_tensor = torch.tensor(features, dtype=torch.float32)
        feat_tensor = feat_tensor.permute(1, 0).unsqueeze(0)
        feat_resized = F.interpolate(
            feat_tensor,
            size=self.num_segments,
            mode='linear',
            align_corners=False
        )
        feat_resized = feat_resized.squeeze(0).permute(1, 0)
        return feat_resized.numpy().astype(np.float32)

    @torch.no_grad()
    def predict(self, features: np.ndarray, apply_smoothing: bool = True) -> Dict:
        """Run inference on video features."""
        input_tensor = self.preprocess_features(features)
        video_score, segment_scores = self.model(input_tensor)

        video_score = video_score.cpu().numpy()[0]
        segment_scores = segment_scores.cpu().numpy()[0]

        if apply_smoothing and self.smoothing_sigma > 0:
            smoothed_scores = gaussian_filter1d(segment_scores, sigma=self.smoothing_sigma)
        else:
            smoothed_scores = segment_scores

        is_anomaly = video_score >= self.threshold

        return {
            'video_score': float(video_score),
            'is_anomaly': bool(is_anomaly),
            'segment_scores': segment_scores.tolist(),
            'smoothed_scores': smoothed_scores.tolist(),
            'max_segment_score': float(segment_scores.max()),
            'anomaly_segments': np.where(segment_scores >= self.threshold)[0].tolist()
        }


# Create detector instance
detector = AnomalyDetector(model)
print("✅ Anomaly Detector initialized!")

✅ Anomaly Detector initialized!


---
## Section 4: Load ALL Features
---

Load features from **ALL** class folders (not just test split).

In [ ]:
"""
Load ALL Features from TimeSformer_Features Folder
"""

def load_all_features(features_path: str = FEATURES_PATH) -> Tuple[List, np.ndarray, List, List]:
    """
    Load ALL extracted features from the complete dataset.

    Returns:
        features: List of feature arrays
        labels: numpy array of labels (0=Normal, 1=Anomaly)
        video_names: List of video names
        class_names: List of class names for each video
    """
    print("\n" + "="*70)
    print("LOADING ALL FEATURES")
    print("="*70)
    print(f"\n📂 Source: {features_path}")

    all_features = []
    all_labels = []
    video_names = []
    class_names = []
    class_counts = defaultdict(int)
    clips_stats = []

    # Get all class folders
    class_folders = [d for d in os.listdir(features_path)
                     if os.path.isdir(os.path.join(features_path, d))]

    print(f"\n📁 Found {len(class_folders)} class folders:")
    print(f"   {sorted(class_folders)}")

    for class_name in sorted(class_folders):
        class_path = os.path.join(features_path, class_name)

        # Label: 0 for Normal, 1 for Anomaly classes
        label = 0 if class_name.lower() == 'normal' else 1

        npy_files = glob.glob(os.path.join(class_path, "*.npy"))

        for file_path in npy_files:
            try:
                features = np.load(file_path)
                clips_stats.append(features.shape[0])

                # Get video name (with extension for matching with raw videos)
                base_name = os.path.splitext(os.path.basename(file_path))[0]

                all_features.append(features)
                all_labels.append(label)
                video_names.append(base_name)
                class_names.append(class_name)
                class_counts[class_name] += 1

            except Exception as e:
                print(f"   ⚠️ Error loading {file_path}: {e}")
                continue

    labels = np.array(all_labels)

    print(f"\n📊 Dataset Statistics:")
    print(f"   Total Videos: {len(all_features)}")
    print(f"   Feature Dimension: 768 (TimeSformer CLS token)")
    print(f"   Clips per video: {min(clips_stats)}-{max(clips_stats)} (avg: {np.mean(clips_stats):.0f})")
    print(f"\n   Anomaly Videos: {np.sum(labels == 1)}")
    print(f"   Normal Videos: {np.sum(labels == 0)}")

    print(f"\n📁 Per-Class Breakdown:")
    for cls in sorted(class_counts.keys()):
        label_type = "Normal" if cls.lower() == 'normal' else "Anomaly"
        print(f"   {cls:20s}: {class_counts[cls]:4d} videos ({label_type})")

    print("="*70)

    return all_features, labels, video_names, class_names


# Load ALL features
all_features, all_labels, video_names, class_names = load_all_features()


LOADING ALL FEATURES

📂 Source: /kaggle/input/timesformer-features/TimeSformer_Features

📁 Found 14 class folders:
   ['Abuse', 'Arrest', 'Arson', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'Normal', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']

📊 Dataset Statistics:
   Total Videos: 1900
   Feature Dimension: 768 (TimeSformer CLS token)
   Clips per video: 1-15257 (avg: 88)

   Anomaly Videos: 950
   Normal Videos: 950

📁 Per-Class Breakdown:
   Abuse               :   50 videos (Anomaly)
   Arrest              :   50 videos (Anomaly)
   Arson               :   50 videos (Anomaly)
   Assault             :   50 videos (Anomaly)
   Burglary            :  100 videos (Anomaly)
   Explosion           :   50 videos (Anomaly)
   Fighting            :   50 videos (Anomaly)
   Normal              :  950 videos (Normal)
   RoadAccidents       :  150 videos (Anomaly)
   Robbery             :  150 videos (Anomaly)
   Shooting            :   50 videos (Anom

---
## Section 5: Run Full Dataset Inference
---

In [ ]:
"""
Run Inference on ALL Videos
"""

def run_full_dataset_inference(
    detector: AnomalyDetector,
    features: List[np.ndarray],
    labels: np.ndarray,
    video_names: List[str],
    class_names: List[str]
) -> Dict:
    """
    Run inference on ALL videos and generate comprehensive results.

    Returns:
        Dictionary with all results keyed by video name
    """
    print("\n" + "="*70)
    print("RUNNING FULL DATASET INFERENCE")
    print("="*70)
    print(f"\n🚀 Processing {len(features)} videos...")

    # Results dictionary keyed by video name (for Phase 5 compatibility)
    all_results = {}

    # Statistics
    class_stats = defaultdict(lambda: {'total': 0, 'detected': 0, 'scores': []})
    correct_predictions = 0
    total_anomalies_detected = 0

    for i, (feat, label, name, cls) in enumerate(tqdm(
        zip(features, labels, video_names, class_names),
        total=len(features),
        desc="Processing videos"
    )):
        # Run inference
        result = detector.predict(feat)

        # Add metadata (including video_name for easy access)
        result['video_name'] = name
        result['ground_truth'] = int(label)
        result['class_name'] = cls
        result['original_clips'] = feat.shape[0]

        # Store by video name (single entry, no duplicate)
        all_results[name] = result

        # Update statistics
        class_stats[cls]['total'] += 1
        class_stats[cls]['scores'].append(result['video_score'])

        if result['is_anomaly']:
            total_anomalies_detected += 1
            class_stats[cls]['detected'] += 1

        # Check if prediction is correct
        gt_is_anomaly = label == 1
        if result['is_anomaly'] == gt_is_anomaly:
            correct_predictions += 1

    # Calculate overall accuracy
    accuracy = correct_predictions / len(features)

    print(f"\n" + "="*70)
    print("INFERENCE COMPLETE")
    print("="*70)
    print(f"\n📊 Overall Statistics:")
    print(f"   Total Videos Processed: {len(features)}")
    print(f"   Anomalies Detected: {total_anomalies_detected}")
    print(f"   Overall Accuracy: {accuracy:.2%}")

    print(f"\n📁 Per-Class Detection Rates:")
    print(f"   {'Class':<20} {'Total':>8} {'Detected':>10} {'Rate':>10} {'Avg Score':>12}")
    print(f"   {'-'*60}")

    for cls in sorted(class_stats.keys()):
        stats = class_stats[cls]
        rate = stats['detected'] / stats['total'] if stats['total'] > 0 else 0
        avg_score = np.mean(stats['scores']) if stats['scores'] else 0
        print(f"   {cls:<20} {stats['total']:>8} {stats['detected']:>10} {rate:>10.1%} {avg_score:>12.4f}")

    print("="*70)

    return all_results, class_stats, accuracy


# Run full inference
all_results, class_stats, overall_accuracy = run_full_dataset_inference(
    detector, all_features, all_labels, video_names, class_names
)


RUNNING FULL DATASET INFERENCE

🚀 Processing 1900 videos...


Processing videos:   0%|          | 0/1900 [00:00<?, ?it/s]


INFERENCE COMPLETE

📊 Overall Statistics:
   Total Videos Processed: 1900
   Anomalies Detected: 912
   Overall Accuracy: 96.63%

📁 Per-Class Detection Rates:
   Class                   Total   Detected       Rate    Avg Score
   ------------------------------------------------------------
   Abuse                      50         46      92.0%       0.9208
   Arrest                     50         46      92.0%       0.9153
   Arson                      50         48      96.0%       0.9515
   Assault                    50         49      98.0%       0.9590
   Burglary                  100         94      94.0%       0.9234
   Explosion                  50         48      96.0%       0.9558
   Fighting                   50         49      98.0%       0.9749
   Normal                    950         13       1.4%       0.0167
   RoadAccidents             150        141      94.0%       0.9195
   Robbery                   150        144      96.0%       0.9477
   Shooting                 

---
## Section 6: Export Results for Phase 5-7
---

In [ ]:
"""
Export Comprehensive Results for Phase 5-7
"""

def export_all_results(
    all_results: Dict,
    class_stats: Dict,
    accuracy: float,
    output_path: str = INFERENCE_OUTPUT_PATH
):
    """
    Export all inference results in formats suitable for Phase 5-7.
    """
    print("\n" + "="*70)
    print("EXPORTING RESULTS")
    print("="*70)

    # 1. Main results file (keyed by video name for Phase 5)
    # This is the PRIMARY output for Phase 5
    results_path = os.path.join(output_path, 'detailed_results.json')
    with open(results_path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f"   ✅ Detailed results: {results_path}")
    print(f"      → {len(all_results)} video entries")

    # 2. Alternative filename for clarity
    all_results_path = os.path.join(output_path, 'all_results.json')
    with open(all_results_path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f"   ✅ All results: {all_results_path}")

    # 3. Summary statistics
    summary = {
        'total_videos': len(all_results),
        'overall_accuracy': accuracy,
        'anomaly_threshold': ANOMALY_THRESHOLD,
        'num_segments': NUM_SEGMENTS,
        'class_statistics': {}
    }

    for cls, stats in class_stats.items():
        summary['class_statistics'][cls] = {
            'total': stats['total'],
            'detected_anomalies': stats['detected'],
            'detection_rate': stats['detected'] / stats['total'] if stats['total'] > 0 else 0,
            'avg_score': float(np.mean(stats['scores'])) if stats['scores'] else 0,
            'min_score': float(np.min(stats['scores'])) if stats['scores'] else 0,
            'max_score': float(np.max(stats['scores'])) if stats['scores'] else 0
        }

    summary_path = os.path.join(output_path, 'inference_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"   ✅ Summary: {summary_path}")

    # 4. Anomaly-only results (filtered for Phase 5)
    anomaly_results = {
        k: v for k, v in all_results.items()
        if v.get('is_anomaly', False)
    }

    anomaly_path = os.path.join(output_path, 'anomaly_results.json')
    with open(anomaly_path, 'w') as f:
        json.dump(anomaly_results, f, indent=2)
    print(f"   ✅ Anomaly-only results: {anomaly_path}")
    print(f"      → {len(anomaly_results)} anomaly detections")

    # 5. CSV for quick inspection
    import csv
    csv_path = os.path.join(output_path, 'all_predictions.csv')

    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            'video_name', 'class_name', 'ground_truth', 'is_anomaly',
            'video_score', 'max_segment_score', 'anomaly_segments'
        ])

        for name, result in all_results.items():
            writer.writerow([
                name,
                result.get('class_name', 'Unknown'),
                'Anomaly' if result.get('ground_truth', 0) == 1 else 'Normal',
                'Anomaly' if result.get('is_anomaly', False) else 'Normal',
                f"{result.get('video_score', 0):.4f}",
                f"{result.get('max_segment_score', 0):.4f}",
                str(result.get('anomaly_segments', []))
            ])
    print(f"   ✅ CSV: {csv_path}")

    print("\n" + "="*70)
    print(f"📁 All results exported to: {output_path}")
    print("="*70)

    return results_path


# Export results
output_json_path = export_all_results(all_results, class_stats, overall_accuracy)


EXPORTING RESULTS
   ✅ Detailed results: /kaggle/working/Inference_Results_All/detailed_results.json
      → 1900 video entries
   ✅ All results: /kaggle/working/Inference_Results_All/all_results.json
   ✅ Summary: /kaggle/working/Inference_Results_All/inference_summary.json
   ✅ Anomaly-only results: /kaggle/working/Inference_Results_All/anomaly_results.json
      → 912 anomaly detections
   ✅ CSV: /kaggle/working/Inference_Results_All/all_predictions.csv

📁 All results exported to: /kaggle/working/Inference_Results_All


---
## Section 7: Verify Output for Phase 5
---

In [ ]:
"""
Verify Output Structure for Phase 5 Compatibility
"""

def verify_phase5_compatibility():
    """Verify that the output is compatible with Phase 5 batch processing."""
    print("\n" + "="*70)
    print("PHASE 5 COMPATIBILITY CHECK")
    print("="*70)

    # Load the exported results
    results_path = os.path.join(INFERENCE_OUTPUT_PATH, 'detailed_results.json')

    with open(results_path, 'r') as f:
        results = json.load(f)

    print(f"\n📂 Loaded: {results_path}")
    print(f"   Total entries: {len(results)}")
    print(f"   Format: DICT (keyed by video_name)")

    # Check required fields for Phase 5
    required_fields = ['video_name', 'video_score', 'is_anomaly', 'segment_scores', 'anomaly_segments', 'class_name']

    # Sample a few entries
    sample_keys = list(results.keys())[:5]

    print(f"\n📋 Sample entries (checking required fields):")
    all_valid = True

    for key in sample_keys:
        entry = results[key]
        missing = [f for f in required_fields if f not in entry]

        if missing:
            print(f"   ❌ {key}: Missing {missing}")
            all_valid = False
        else:
            print(f"   ✅ {key}")
            print(f"      Score: {entry['video_score']:.4f}, Anomaly: {entry['is_anomaly']}, Class: {entry['class_name']}")
            print(f"      Segments: {len(entry['segment_scores'])}, Anomaly Segs: {entry['anomaly_segments']}")

    # Count anomalies per class (for Phase 5 stratified sampling)
    print(f"\n📊 Anomalies available for Phase 5 (10 per class target):")
    class_anomalies = defaultdict(int)

    for key, entry in results.items():
        if entry.get('is_anomaly', False):
            cls = entry.get('class_name', 'Unknown')
            class_anomalies[cls] += 1

    print(f"   {'Class':<20} {'Anomalies':>12} {'Status':>15}")
    print(f"   {'-'*47}")

    for cls in sorted(class_anomalies.keys()):
        count = class_anomalies[cls]
        status = "✅ Sufficient" if count >= 10 else f"⚠️ Only {count}"
        print(f"   {cls:<20} {count:>12} {status:>15}")

    total_anomalies = sum(class_anomalies.values())
    print(f"\n   Total anomalies available: {total_anomalies}")
    print(f"   Phase 5 target: 130 videos (10 × 13 classes)")

    print("\n" + "="*70)
    if all_valid:
        print("✅ OUTPUT IS READY FOR PHASE 5!")
    else:
        print("❌ Some entries have issues - please check")
    print("="*70)


# Run verification
verify_phase5_compatibility()


PHASE 5 COMPATIBILITY CHECK

📂 Loaded: /kaggle/working/Inference_Results_All/detailed_results.json
   Total entries: 1900
   Format: DICT (keyed by video_name)

📋 Sample entries (checking required fields):
   ✅ Abuse037_x264
      Score: 0.9973, Anomaly: True, Class: Abuse
      Segments: 16, Anomaly Segs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 14, 15]
   ✅ Abuse046_x264
      Score: 1.0000, Anomaly: True, Class: Abuse
      Segments: 16, Anomaly Segs: [1, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
   ✅ Abuse044_x264
      Score: 1.0000, Anomaly: True, Class: Abuse
      Segments: 16, Anomaly Segs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
   ✅ Abuse025_x264
      Score: 1.0000, Anomaly: True, Class: Abuse
      Segments: 16, Anomaly Segs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
   ✅ Abuse024_x264
      Score: 1.0000, Anomaly: True, Class: Abuse
      Segments: 16, Anomaly Segs: [1, 5, 10, 11, 12, 13, 14, 15]

📊 Anomalies available for Phase 5 (10 per class 

---
## Section 8: Summary
---

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║           PHASE 4B: FULL DATASET INFERENCE COMPLETE              ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  ✅ Processed ALL videos in the dataset                          ║
║                                                                  ║
║  OUTPUT FILES:                                                   ║
║  ├── detailed_results.json  ← PRIMARY (for Phase 5)              ║
║  ├── all_results.json       ← Alias                              ║
║  ├── anomaly_results.json   ← Anomaly-only subset                ║
║  ├── inference_summary.json ← Statistics                         ║
║  └── all_predictions.csv    ← Quick inspection                   ║
║                                                                  ║
║  JSON STRUCTURE (per video):                                     ║
║  {                                                               ║
║    "video_score": 0.94,                                          ║
║    "is_anomaly": true,                                           ║
║    "segment_scores": [...],  # 16 values                         ║
║    "smoothed_scores": [...], # 16 values                         ║
║    "anomaly_segments": [10, 11, 12, 13],                         ║
║    "max_segment_score": 0.96,                                    ║
║    "ground_truth": 1,                                            ║
║    "class_name": "Fighting"                                      ║
║  }                                                               ║
║                                                                  ║
╠══════════════════════════════════════════════════════════════════╣
║  PHASE 5 USAGE:                                                  ║
║  ├── Load: detailed_results.json                                 ║
║  ├── Filter: is_anomaly == True                                  ║
║  ├── Sample: 10 videos per class                                 ║
║  └── Extract: Before/Peak/After frames                           ║
╠══════════════════════════════════════════════════════════════════╣
║  NEXT: Run Phase 5 notebook with these results!                  ║
╚══════════════════════════════════════════════════════════════════╝
""")

print(f"\n📁 Output Location: {INFERENCE_OUTPUT_PATH}")
print(f"📊 Total Videos: {len(all_features)}")
print(f"🎯 Accuracy: {overall_accuracy:.2%}")


╔══════════════════════════════════════════════════════════════════╗
║           PHASE 4B: FULL DATASET INFERENCE COMPLETE              ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  ✅ Processed ALL videos in the dataset                          ║
║                                                                  ║
║  OUTPUT FILES:                                                   ║
║  ├── detailed_results.json  ← PRIMARY (for Phase 5)              ║
║  ├── all_results.json       ← Alias                              ║
║  ├── anomaly_results.json   ← Anomaly-only subset                ║
║  ├── inference_summary.json ← Statistics                         ║
║  └── all_predictions.csv    ← Quick inspection                   ║
║                                                                  ║
║  JSON STRUCTURE (per video):                                     ║
║  {                              